# 08 — Listing Lens V4: AI Generator Adaptation

This notebook documents the V4 experiment using the existing EfficientNetB0 V3 checkpoint as the starting point. It does **not** retrain from scratch and does not modify the V3 model or active Streamlit configuration.

The intake contains 54 AI candidate images (GPT 26, Gemini 19, Meta AI 9). 37 are allocated to fine-tuning, 8 to validation and 9 to a generator-challenge test. Similarity groups remain together. The 17-image holdout is exploratory and too small for strong generalisation claims.

**Prerequisites:** the existing Dataset V2 split folder, the Listing Lens repository (including `models_v3/car_cnn_model_v3.keras`), TensorFlow 2.21.0, Pillow, ImageHash and scikit-learn. Read `README_V4.md` first.

## 1. Configure paths
Update `V2_ROOT` to the Dataset V2 split folder on your computer. Run this notebook from the Listing Lens repository root, or adjust the script paths.

In [ ]:
from pathlib import Path
import json, subprocess, sys

PROJECT_ROOT = Path.cwd()
V2_ROOT = Path("/absolute/path/to/car_dataset_split_v1").expanduser()  # EDIT THIS
V4_ROOT = PROJECT_ROOT / "data_v4"
STARTER = PROJECT_ROOT / "v4_training"  # folder containing these scripts and ai_candidates/

assert (PROJECT_ROOT / "models_v3/car_cnn_model_v3.keras").is_file(), "V3 model not found; open notebook from Listing Lens project root."
assert V2_ROOT.is_dir(), f"Set V2_ROOT to your existing Dataset V2 folder: {V2_ROOT}"
assert (STARTER / "prepare_v4_dataset.py").is_file(), "Place this starter folder at <project>/v4_training or update STARTER."
print("Project:", PROJECT_ROOT)
print("Dataset V2:", V2_ROOT)

## 2. Prepare the V4 dataset
This copies Dataset V2 into a new directory and adds the allocated new AI candidates only to the `ai_generated` class. The original dataset folder is not modified.

In [ ]:
cmd = [sys.executable, str(STARTER / "prepare_v4_dataset.py"), "--v2-root", str(V2_ROOT), "--output-root", str(V4_ROOT)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
summary = json.loads((V4_ROOT / "v4_preparation_summary.json").read_text())
summary

## 3. Review duplicates and counts before training
Check exact duplicates skipped and cross-dataset perceptual near-matches. Resolve any real/AI label conflicts before proceeding.

In [ ]:
for filename in ["v4_exact_duplicates_skipped.csv", "v4_cross_dataset_near_duplicate_review.csv", "v4_additions_manifest.csv"]:
    path = V4_ROOT / filename
    print(f"\n--- {filename} ---")
    print(path.read_text()[:5000] if path.exists() else "Not created")

## 4. Fine-tune V3 into a separate V4 candidate
Phase 1 adapts the classifier head with a frozen backbone. Phase 2 unfreezes only the top 40 EfficientNetB0 layers while keeping Batch Normalization frozen. The script verifies the V3 SHA-256 and never overwrites the source checkpoint.

In [ ]:
train_script = STARTER / "train_v4.py"
cmd = [sys.executable, str(train_script), "--dataset-root", str(V4_ROOT),
       "--v3-model", str(PROJECT_ROOT / "models_v3/car_cnn_model_v3.keras"),
       "--v3-config", str(PROJECT_ROOT / "config/cnn_inference_config.json"),
       "--output-dir", str(PROJECT_ROOT / "outputs_v4"), "--batch-size", "8"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

## 5. Inspect V4 report
Do not update the Streamlit model path yet. First compare V3 and V4 on the same original V2 test set, inspect false positives, and treat generator-specific rates from 4/3/2 test images as preliminary only.

In [ ]:
report_path = PROJECT_ROOT / "outputs_v4/v4_training_report.json"
report = json.loads(report_path.read_text())
print(json.dumps(report, indent=2))